# bench_imai — CIFAKE vs DeepDetect Vision Benchmark

This notebook refreshes the CIFAKE benchmark by routing every experiment through
`BenchmarkRunner` and the shared `pipelines_torch` registries. We compare the
modern `timm` backbones introduced in `vision_models.py`, then reuse the saved
weights to audit generalisation on the DeepDetect 2025 dataset.

 GPU is optional but recommended for the heavier backbones


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname,_,_ in os.walk('/kaggle/input'):
    print (dirname)
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## 1. Environment setup
Install optional vision dependencies (`timm`, `kaggle`) so the registry backbones
load without manual package management.


In [ ]:
%rm -rf bench_research_ml_project
!git clone https://github.com/oremaz/bench_research_ml_project
%cd bench_research_ml_project
!pip install -r requirements.txt
!pip uninstall numpy scipy scikit-learn -y
!pip install numpy==1.24.3 scipy==1.10.1 scikit-learn==1.3.0
!pip install -q timm==1.0.9 kaggle rich

In [2]:
%cd ml_pipeline

/kaggle/working/bench_research_ml_project/ml_pipeline


In [3]:
%cd third_party/FatFormer
%mkdir pretrained
%cd pretrained
!wget https://openaipublic.azureedge.net/clip/models/b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836/ViT-L-14.pt
%ls pretrained/
%cd ../../..

/kaggle/working/bench_research_ml_project/ml_pipeline/third_party/FatFormer
/kaggle/working/bench_research_ml_project/ml_pipeline/third_party/FatFormer/pretrained
--2025-10-05 11:13:52--  https://openaipublic.azureedge.net/clip/models/b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836/ViT-L-14.pt
Resolving openaipublic.azureedge.net (openaipublic.azureedge.net)... 13.107.213.38, 13.107.246.38, 2620:1ec:bdf::38, ...
Connecting to openaipublic.azureedge.net (openaipublic.azureedge.net)|13.107.213.38|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 932768134 (890M) [application/octet-stream]
Saving to: ‘ViT-L-14.pt’

ViT-L-14.pt         100%[===================>] 889.56M  46.0MB/s    in 18s     

2025-10-05 11:14:10 (49.2 MB/s) - ‘ViT-L-14.pt’ saved [932768134/932768134]

ls: cannot access 'pretrained/': No such file or directory
/kaggle/working/bench_research_ml_project/ml_pipeline


## 2. Imports and deterministic utilities
Everything important (models, metrics, benchmarking) is imported from the shared
library so we avoid redefining models or augmentations inside the notebook.


In [12]:
import os
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

from pipelines_torch.benchmark import BenchmarkRunner
from pipelines_torch.vision_models import MODEL_REGISTRY as VISION_MODELS
from pipelines_torch.base import SimplePredictor
from utils.metrics import METRIC_REGISTRY
from utils.utils import load_model

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 3. Download and prepare CIFAKE
The helper below pulls CIFAKE from Kaggle and converts the PyTorch `ImageFolder`
structure into NumPy arrays so that `BenchmarkRunner` can operate on it.


In [5]:
from pathlib import Path

from utils.kaggle_utils import ensure_kaggle_dataset

KAGGLE_DATASET = "birdy654/cifake-real-and-ai-generated-synthetic-images"
DATA_DIR = ensure_kaggle_dataset(
    dataset_slug=KAGGLE_DATASET,
    local_dir=Path("data/cifake"),
    description="CIFAKE dataset",
    kaggle_subdir="cifake-real-and-ai-generated-synthetic-images",
)

if (DATA_DIR / "train").exists():
    print(f"✅ CIFAKE data available at {DATA_DIR}")
else:
    print(f"⚠️ CIFAKE dataset missing expected 'train' directory at {DATA_DIR}")

✅ Using Kaggle input for CIFAKE dataset at /kaggle/input/cifake-real-and-ai-generated-synthetic-images
✅ CIFAKE data available at /kaggle/input/cifake-real-and-ai-generated-synthetic-images


In [6]:
IMG_SIZE = 32
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dir = DATA_DIR / "train"
val_dir = DATA_DIR / "test"
if not train_dir.exists() or not val_dir.exists():
    raise FileNotFoundError("Expected CIFAKE to expose train/ and test/ splits")

train_ds = datasets.ImageFolder(train_dir, transform=transform)
val_ds = datasets.ImageFolder(val_dir, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

class_names = train_ds.classes
num_classes = len(class_names)
print(f"Classes: {class_names}")

Classes: ['FAKE', 'REAL']


In [ ]:
def dataset_to_numpy(dataset: datasets.ImageFolder):
    tensors = [img for img, _ in dataset]
    X = torch.stack(tensors).numpy()
    y = np.array(dataset.targets, dtype=np.int64)
    return X.astype(np.float32), y

X_all, y_all = dataset_to_numpy(train_ds)
X_test, y_test = dataset_to_numpy(val_ds)

In [49]:
print(f"Test shape: {X_test.shape}")

Test shape: (20000, 3, 32, 32)


## 4. Configure `BenchmarkRunner`
We select the modern `timm` backbones registered in `vision_models.py` and
evaluate macro metrics from `utils.metrics`.


In [8]:
models = []
for model in VISION_MODELS: 
    if model not in ["qwen2_vl_qlora"]: 
        models.append(model)

model_configs = []
epochs = {}
for name in models:
    if name not in VISION_MODELS:
        raise KeyError(f"{name} is not registered in pipelines_torch.vision_models")
    model_configs.append({
        "name": name,
        "class": VISION_MODELS[name],
        "params": {"num_classes": num_classes},
    })
    epochs[name] = 7


runner = BenchmarkRunner(
    model_configs=model_configs,
    augmentations=[None],
    task_type="classification",
    device=DEVICE,
    epochs=epochs,
    batch_size=32,
    early_stopping=None,
    use_class_weights=True,
    use_kfold=False,
    learning_rate=3e-4,
    path_start="bench_imai",
    random_state=SEED,
)
results_df = runner.run(X_all, y_all)
results_df


Augmentations:   0%|          | 0/1 [00:00<?, ?it/s]


Running Model: diffusionfake_official | Augmentation: none
Class distribution: {0: 40000, 1: 40000}


/kaggle/working/bench_research_ml_project/ml_pipeline/third_party/DiffusionFake/models/image.py:75: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  self.encoder = encoder_params[encoder]["init_op"](**kwargs)


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

/kaggle/working/bench_research_ml_project/ml_pipeline/pipelines_torch/base.py:702: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler()


...

Epoch 6/7 - Train Loss: 0.0439, Val Loss: 0.0946, F1: 0.9672, ROC-AUC: 0.9953, PR-AUC: 0.9940


/kaggle/working/bench_research_ml_project/ml_pipeline/pipelines_torch/base.py:1167: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with self.autocast():


Epoch 7/7 - Train Loss: 0.0367, Val Loss: 0.0892, F1: 0.9665, ROC-AUC: 0.9956, PR-AUC: 0.9959
Selected best epoch: 5 based on selection metric



Models: 100%|██████████| 1/1 [19:13<00:00, 1153.69s/it]       

Computed class weights: tensor([1., 1.], device='cuda:0')


In [9]:
import os
import tarfile
from IPython.display import FileLink

# Change to the results directory
os.chdir('/kaggle/working/results')

# Create tar.gz archive containing all .pt files
with tarfile.open('/kaggle/working/all_files.tar.gz', 'w:gz') as tar:
    for root, dirs, files in os.walk('.'):
        for file in files:
            tar.add(os.path.join(root, file))

# Generate download link
os.chdir('/kaggle/working')
FileLink('all_files.tar.gz')


/kaggle/working/all_files.tar.gz

## 6. Evaluate the CIFAKE test split
Re-use the same helper to score the official CIFAKE `test/` directory.


In [37]:
import torch
from torch.utils.data import DataLoader, Dataset, TensorDataset
from typing import Callable, List, Optional, Dict, Any, Tuple, Union
import numpy as np
import random
from functools import partial
from collections.abc import Sized
from typing import cast
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
from sklearn.preprocessing import label_binarize
from utils.utils import select_best_epoch


class Evaluate:
    """Base utilities shared across torch and sklearn pipelines."""

    def __init__(self) -> None:
        # These attributes should be set by subclasses
        self.model = None
        self.task_type = "classification"
        self.metrics = []
        self.batch_size = 32
        self.random_state = 42
        self.device = "cpu"

    def _default_selection_metric(self) -> str:
        """Return the metric name used to pick the best epoch."""
        return "roc_auc" if self.task_type == "classification" else "r2_score"

    def prepare_data_internal(
        self,
        X: np.ndarray,
        y: Optional[np.ndarray] = None,
        *,
        train: bool = True,
    ) -> DataLoader:
        """Subclasses must implement how raw arrays become loaders."""
        raise NotImplementedError("Subclasses must implement prepare_data_internal")

    # ----- Evaluation helpers -------------------------------------------------
    def evaluate(self, X: np.ndarray, y: np.ndarray) -> Dict[str, float]:
        if X is None or y is None:
            return {}
        if hasattr(self, "model") and hasattr(self.model, "eval"):
            return self._evaluate_torch(X, y)
        return self._evaluate_sklearn(X, y)

    def _evaluate_torch(self, X: np.ndarray, y: np.ndarray) -> Dict[str, float]:
        loader: DataLoader = self.prepare_data_internal(X, y, train=False)
        self.model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for batch in loader:
                xb, yb = batch
                xb = xb.to(self.device)
                yb = yb.to(self.device)
                logits = self.model(xb)
                if self.task_type == "regression":
                    outputs = logits.squeeze(-1)
                else:
                    outputs = torch.softmax(logits, dim=1)
                preds.append(outputs.cpu())
                targets.append(yb.cpu())
        preds_np = torch.cat(preds).numpy()
        targets_np = torch.cat(targets).numpy()
        results: Dict[str, float] = {}
        for metric in self.metrics:
            key = getattr(metric, "name", None) or getattr(metric, "__name__", None) or str(metric)
            try:
                results[key] = float(metric(targets_np, preds_np))
            except Exception:
                results[key] = 0.0
        return results

    def _evaluate_sklearn(self, X: np.ndarray, y: np.ndarray) -> Dict[str, float]:
        if not hasattr(self.model, "predict"):
            raise AttributeError(f"Model of type {type(self.model)} does not have a predict method.")
        y_pred = self.model.predict(X)
        results: Dict[str, float] = {}
        for metric in self.metrics:
            key = getattr(metric, "name", None) or getattr(metric, "__name__", None) or str(metric)
            try:
                results[key] = float(metric(y, y_pred))
            except Exception:
                results[key] = 0.0
        return results

    # ----- Prediction helpers -------------------------------------------------
    def predict(self, X: np.ndarray) -> np.ndarray:
        if hasattr(self, "model") and hasattr(self.model, "eval"):
            return self._predict_torch(X)
        return self._predict_sklearn(X)

    def _predict_torch(self, X: np.ndarray) -> np.ndarray:
        loader: DataLoader = self.prepare_data_internal(X, train=False)
        self.model.eval()
        preds = []
        with torch.no_grad():
            for batch in loader:
                xb = batch[0] if isinstance(batch, (list, tuple)) else batch
                xb = xb.to(self.device)
                logits = self.model(xb)
                if self.task_type == "regression":
                    preds.append(logits.squeeze(-1).cpu())
                else:
                    if logits.ndim == 1:
                        logits = logits.unsqueeze(-1)
                    if logits.shape[1] == 1:
                        probs = torch.sigmoid(logits).squeeze(-1)
                        preds.append((probs > 0.5).long().cpu())
                    else:
                        preds.append(torch.softmax(logits, dim=1).argmax(dim=1).cpu())
        return torch.cat(preds).numpy()

    def _predict_sklearn(self, X: np.ndarray) -> np.ndarray:
        if hasattr(self.model, "predict"):
            return self.model.predict(X)
        raise AttributeError(f"Model of type {type(self.model)} does not have a predict method.")

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        if self.task_type != "classification":
            raise ValueError("predict_proba is only available for classification tasks")
        if hasattr(self, "model") and hasattr(self.model, "eval"):
            return self._predict_proba_torch(X)
        return self._predict_proba_sklearn(X)

    def _predict_proba_torch(self, X: np.ndarray) -> np.ndarray:
        loader: DataLoader = self.prepare_data_internal(X, train=False)
        self.model.eval()
        probs_list = []
        with torch.no_grad():
            for batch in loader:
                xb = batch[0] if isinstance(batch, (list, tuple)) else batch
                xb = xb.to(self.device)
                logits = self.model(xb)
                if logits.ndim == 1:
                    logits = logits.unsqueeze(-1)
                if logits.shape[1] == 1:
                    pos = torch.sigmoid(logits).squeeze(-1)
                    probs = torch.stack((1 - pos, pos), dim=1)
                else:
                    probs = torch.softmax(logits, dim=1)
                probs_list.append(probs.cpu())
        return torch.cat(probs_list).numpy()

    def _predict_proba_sklearn(self, X: np.ndarray) -> np.ndarray:
        if hasattr(self.model, "predict_proba"):
            return self.model.predict_proba(X)
        raise ValueError("Model does not support predict_proba")


class ComputeScore:
    """Utility scoring helpers shared by torch and sklearn pipelines."""

    @staticmethod
    def f1(targets: np.ndarray, preds: np.ndarray, task_type: str) -> Optional[float]:
        if task_type != "classification":
            return None
        try:
            targets_array = np.asarray(targets)
            preds_array = np.asarray(preds)
            average = "binary" if len(np.unique(targets_array)) == 2 else "macro"
            if preds_array.ndim > 1:
                preds_processed = np.argmax(preds_array, axis=1)
            else:
                if len(np.unique(targets_array)) == 2:
                    preds_processed = (preds_array >= 0.5).astype(int)
                else:
                    preds_processed = preds_array
            return float(f1_score(targets_array, preds_processed, average=average, zero_division=0))
        except Exception:
            return None

    @staticmethod
    def r2(targets: np.ndarray, preds: np.ndarray, task_type: str) -> Optional[float]:
        if task_type != "regression":
            return None
        try:
            from sklearn.metrics import r2_score
            return float(r2_score(targets, preds))
        except Exception:
            return None

    @staticmethod
    def roc_auc(targets: np.ndarray, preds: np.ndarray, task_type: str) -> Optional[float]:
        if task_type != "classification":
            return None
        try:
            y_true = np.asarray(targets)
            y_scores = np.asarray(preds)
            if y_scores.ndim == 1:
                return float(roc_auc_score(y_true, y_scores))
            if y_scores.ndim == 2:
                if y_scores.shape[1] == 1:
                    return float(roc_auc_score(y_true, y_scores[:, 0]))
                if y_scores.shape[1] == 2:
                    return float(roc_auc_score(y_true, y_scores[:, 1]))
                return float(roc_auc_score(y_true, y_scores, multi_class="ovr", average="macro"))
            return None
        except Exception:
            return None

    @staticmethod
    def pr_auc(targets: np.ndarray, preds: np.ndarray, task_type: str) -> Optional[float]:
        if task_type != "classification":
            return None
        try:
            y_true = np.asarray(targets)
            y_scores = np.asarray(preds)
            if y_scores.ndim == 1:
                return float(average_precision_score(y_true, y_scores))
            if y_scores.ndim == 2:
                if y_scores.shape[1] == 1:
                    return float(average_precision_score(y_true, y_scores[:, 0]))
                if y_scores.shape[1] == 2:
                    return float(average_precision_score(y_true, y_scores[:, 1]))
                classes = np.arange(y_scores.shape[1])
                y_true_bin = label_binarize(y_true, classes=classes)
                return float(average_precision_score(y_true_bin, y_scores, average="macro"))
            return None
        except Exception:
            return None
        

class SimplePredictor(Evaluate):
    """
    Lightweight class for making predictions with trained models.
    Minimal setup required - just provide the model and basic info.
    """
    
    def __init__(self, model, task_type: str = "classification", device: str = "cpu", batch_size: int = 32):
        super().__init__()
        self.model = model
        self.task_type = task_type
        self.device = device
        self.batch_size = batch_size
        self.random_state = 42
        self.metrics = []
        
        # Move PyTorch models to device
        if hasattr(model, 'to'):
            self.model = model.to(device)
    
    def prepare_data_internal(self, X: np.ndarray, y: Optional[np.ndarray] = None, train: bool = True) -> DataLoader:
        """Prepare data for PyTorch models, return None for sklearn models."""
        if hasattr(self, 'device') and hasattr(self.model, 'eval'):
            # PyTorch model - create DataLoader
            X_tensor = torch.tensor(X, dtype=torch.float32)
            if y is not None:
                y_tensor = torch.tensor(y, dtype=torch.float32 if self.task_type == "regression" else torch.long)
                dataset = TensorDataset(X_tensor, y_tensor)
            else:
                dataset = TensorDataset(X_tensor)
            
            return DataLoader(
                dataset, 
                batch_size=self.batch_size, 
                shuffle=False,  # No need to shuffle for prediction
                num_workers=0,
                drop_last=False
            )
        else:
            # sklearn model - return None (data passed directly)
            return None



In [42]:
def evaluate_saved_models(model_names: Iterable[str], X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    metric_map = {
        "accuracy": "accuracy",
        "f1_macro": "f1",
        "precision_macro": "precision",
        "recall_macro": "recall",
        "roc_auc": "roc_auc",
        "pr_auc": "pr_auc",
    }
    records = []
    for name in model_names:
        try:
            model = load_model(VISION_MODELS[name], name, {"num_classes": num_classes}, path_start="bench-imai-final/bench_imai")
        except FileNotFoundError:
            print(f"⚠️ Skipping {name}: checkpoint not found")
            continue
        print(name)
        predictor = SimplePredictor(model, task_type="classification", device=DEVICE, batch_size=32)
        probs = predictor.predict_proba(X)
        scores = {
            label: float(METRIC_REGISTRY[key](y, probs))
            for label, key in metric_map.items()
        }
        records.append({
            "model": name,
            **scores,
        })
    return pd.DataFrame.from_records(records)

In [43]:
models = []
for model in VISION_MODELS: 
    if model not in ["qwen2_vl_qlora", "fatformer_official"]: 
        models.append(model)
cifake_test_metrics = evaluate_saved_models(models, X_test, y_test)
cifake_test_metrics.sort_values("accuracy", ascending=False)

simple_cnn
adaptive_cnn
residual_cnn
resnet50


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

timm_vit_base_patch16


model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

timm_vit_mae_base


,model,accuracy,f1_macro,precision_macro,recall_macro,roc_auc,pr_auc
5,diffusionfake_official,0.97105,0.971049,0.971126,0.97105,0.995888,0.996081
1,adaptive_cnn,0.96200,0.962000,0.962024,0.96200,0.993488,0.993793
6,timm_convnextv2_tiny,0.96030,0.960292,0.960651,0.96030,0.993439,0.993835
2,residual_cnn,0.95795,0.957949,0.958015,0.95795,0.992116,0.992437
4,clip_classifier,0.94625,0.946250,0.946254,0.94625,0.988036,0.988249
0,simple_cnn,0.94280,0.942800,0.942800,0.94280,0.986278,0.986954
7,timm_efficientnetv2_s,0.92165,0.921646,0.921731,0.92165,0.978299,0.979340
9,timm_vit_mae_base,0.90420,0.904047,0.906790,0.90420,0.970198,0.971360
3,resnet50,0.90185,0.901845,0.901935,0.90185,0.964649,0.965742
8,timm_vit_base_patch16,0.87950,0.879249,0.882683,0.87950,0.954726,0.956387


## 7. shoes-dataset 2025 generalisation check
The repository previously evaluated CIFAKE models on the shoes-dataset.
We keep that workflow: download the dataset from Kaggle (if necessary), locate an
`ImageFolder`-compatible split, and score it with the same predictor helper.

In [56]:
from pathlib import Path
from utils.kaggle_utils import ensure_kaggle_dataset

# Kaggle dataset: https://www.kaggle.com/datasets/sunnykakar/shoes-dataset-real-and-ai-generated-images
KAGGLE_SHOES = "sunnykakar/shoes-dataset-real-and-ai-generated-images"

# ✅ Use the actual extracted top-level folder name from the dataset zip:
EXPECTED_TOPDIR = "shoes-dataset-real-and-ai-generated-images"

SHOES_DIR = ensure_kaggle_dataset(
    dataset_slug=KAGGLE_SHOES,
    local_dir=Path("data/shoes-ai-vs-real"),
    description="SunnyKakar Shoes (AI vs Real)",
    kaggle_subdir=EXPECTED_TOPDIR,   # <— was "shoes-dataset" before; that's wrong for this dataset
)

if SHOES_DIR.exists():
    print(f"✅ Shoes dataset available at {SHOES_DIR}")
else:
    print(f"⚠️ Shoes dataset missing at {SHOES_DIR}")

def is_imagefolder_dir(path: Path) -> bool:
    path = Path(path)
    if not path.is_dir():
        return False
    subdirs = [p for p in path.iterdir() if p.is_dir()]
    if len(subdirs) < 2:
        return False
    # At least one class subdir must contain files
    return any(any(child.is_file() for child in d.iterdir()) for d in subdirs)

def find_imagefolder_split(base_dir: Path) -> Path:
    base_dir = Path(base_dir)

    # If the provided base is already an ImageFolder root, use it.
    if is_imagefolder_dir(base_dir):
        return base_dir

    # Common split folder names
    preferred = ("train", "test", "validation", "val", "eval", "holdout")
    for name in preferred:
        for variant in {name, name.upper(), name.capitalize()}:
            candidate = base_dir / variant
            if is_imagefolder_dir(candidate):
                return candidate

    # Extra: If ensure_kaggle_dataset returned local_dir but the data is nested one level deeper,
    # look for a folder that contains both 'ai-midjourney' and 'real' with images inside.
    for candidate in base_dir.glob("*"):
        if candidate.is_dir():
            ai_dir = candidate / "ai-midjourney"
            real_dir = candidate / "real"
            if ai_dir.exists() and real_dir.exists() and is_imagefolder_dir(candidate):
                return candidate

    # General recursive search as a final fallback.
    for candidate in sorted(base_dir.rglob("*")):
        if is_imagefolder_dir(candidate):
            return candidate

    raise ValueError(f"Could not locate an ImageFolder split inside {base_dir}")

IMG_SIZE = 32
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

models = []
for model in VISION_MODELS: 
    if model not in ["qwen2_vl_qlora", "fatformer_official"]: 
        models.append(model)

def evaluate_imagefolder_split(split_dir: Path, dataset_name: str) -> pd.DataFrame:
    dataset = datasets.ImageFolder(split_dir, transform=transform)
    if len(dataset) == 0:
        raise ValueError(f"No samples detected under {split_dir}")
    X_eval, y_eval = dataset_to_numpy(dataset)
    df = evaluate_saved_models(models, X_eval, y_eval)
    return df

shoes_split = find_imagefolder_split(SHOES_DIR)
print(f"Using {shoes_split} for evaluation")
shoes_metrics = evaluate_imagefolder_split(shoes_split, "Shoes: Real vs AI (SunnyKakar)")
shoes_metrics.sort_values("accuracy", ascending=False)

✅ Using Kaggle input for SunnyKakar Shoes (AI vs Real) at /kaggle/input/shoes-dataset-real-and-ai-generated-images
✅ Shoes dataset available at /kaggle/input/shoes-dataset-real-and-ai-generated-images
Using /kaggle/input/shoes-dataset-real-and-ai-generated-images for evaluation
simple_cnn
adaptive_cnn
residual_cnn
resnet50
clip_classifier


/kaggle/working/bench_research_ml_project/ml_pipeline/third_party/DiffusionFake/models/image.py:75: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  self.encoder = encoder_params[encoder]["init_op"](**kwargs)


diffusionfake_official
Dropping unsupported 'img_size' argument for model convnextv2_tiny.fcmae_ft_in1k
timm_convnextv2_tiny
Dropping unsupported 'img_size' argument for model tf_efficientnetv2_s.in21k
timm_efficientnetv2_s
timm_vit_base_patch16
timm_vit_mae_base


,model,accuracy,f1_macro,precision_macro,recall_macro,roc_auc,pr_auc
4,clip_classifier,0.791380,0.785129,0.782183,0.797102,0.872095,0.843917
3,resnet50,0.718936,0.712513,0.712838,0.725552,0.806164,0.732923
8,timm_vit_base_patch16,0.640532,0.640286,0.693452,0.687656,0.849343,0.811462
5,diffusionfake_official,0.601100,0.599566,0.668542,0.654996,0.786789,0.710885
6,timm_convnextv2_tiny,0.589638,0.587304,0.662844,0.646252,0.824511,0.797509
7,timm_efficientnetv2_s,0.589179,0.584995,0.678499,0.652291,0.810132,0.722603
9,timm_vit_mae_base,0.563044,0.557688,0.652172,0.627002,0.807496,0.729332
1,adaptive_cnn,0.552499,0.546098,0.644470,0.618046,0.808908,0.795304
2,residual_cnn,0.521320,0.507837,0.636216,0.597007,0.786329,0.755712
0,simple_cnn,0.507107,0.497472,0.596625,0.574897,0.771690,0.757223
